# 标签扩展：更稳健的标签传播
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：04_半监督学习 | 源文件：标签扩展.py | 核心功能：LabelSpreading 正则化、alpha 参数影响、标签比例实验

## 概述

标签扩展（Label Spreading）是标签传播的正则化版本。通过 alpha 参数控制"传播"和"保留原始标签"之间的平衡，解决了标准标签传播的数值不稳定性问题。

核心公式：Y_final = (1-alpha) x Y_propagated + alpha x Y_initial。alpha 越小，传播越强；alpha 越大，越保守。

脚本系统地对比了不同 alpha 值、标签比例和邻居数的影响。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score, classification_report
# --- 导入库 ---
from sklearn.neighbors import KNeighborsClassifier

## 数学原理

### 1. Label Spreading 的正则化框架

**代码对应**：`LabelSpreading(kernel="knn", alpha=0.2)` 中 alpha 控制正则化。

Label Spreading 最小化以下能量函数：

$$\min_{\mathbf{F}} \sum_{i,j}W_{ij}\left\|\frac{\mathbf{F}_i}{\sqrt{d_i}} - \frac{\mathbf{F}_j}{\sqrt{d_j}}\right\|^2 + \mu\sum_i\|\mathbf{F}_i - \mathbf{Y}_i\|^2$$

第一项（平滑项）：鼓励图上相邻节点的标签相似（标签沿图平滑传播）。

第二项（拟合项）：鼓励预测标签接近初始标签。

### 2. 闭式解

对称归一化拉普拉斯 $\mathbf{L}_{\text{sym}} = \mathbf{I} - \mathbf{D}^{-1/2}\mathbf{W}\mathbf{D}^{-1/2}$，最优解为：

$$(\mathbf{L}_{\text{sym}} + \mu\mathbf{I})\mathbf{F} = \mu\mathbf{Y}$$

sklearn 使用迭代形式：

$$\mathbf{F}^{(t+1)} = \alpha\mathbf{S}\mathbf{F}^{(t)} + (1-\alpha)\mathbf{Y}$$

其中 $\mathbf{S} = \mathbf{D}^{-1/2}\mathbf{W}\mathbf{D}^{-1/2}$，$\alpha = 1/(1+\mu)$。

**代码对应**：`alpha=0.2` 意味着 $\mu = (1-\alpha)/\alpha = 4$，拟合项权重较大。

### 3. alpha 的直觉

$$\mathbf{F}^* = (1-\alpha)(\mathbf{I} - \alpha\mathbf{S})^{-1}\mathbf{Y}$$

- $\alpha \to 0$：$\mathbf{F}^* \approx \mathbf{Y}$，几乎不传播（只用原始标签）
- $\alpha \to 1$：完全传播，类似 LabelPropagation
- $\alpha = 0.2$：平衡传播和原始标签

**代码对应**：代码中 `for alpha in [0.001, 0.1, ..., 0.999]` 展示了 alpha 对准确率的影响。

### 4. 与图正则化的联系

Label Spreading 本质上是**图正则化半监督学习**（Graph-Regularized Semi-Supervised Learning）的特例。一般形式：

$$\min_f \sum_{i \in L}\ell(f(\mathbf{x}_i), y_i) + \lambda\sum_{i,j}W_{ij}(f(\mathbf{x}_i) - f(\mathbf{x}_j))^2$$

第一项是有标签数据上的损失，第二项是图上的平滑正则化。$\lambda$ 控制两者权衡。

### 5. 标签比例的影响

**代码对应**：代码中 `for ratio in [0.05, ..., 1.0]` 展示了标签比例对效果的影响。

标签越多，传播的"锚点"越多，效果越好。但即使只有 10-15% 的标签，Label Spreading 也能通过图结构显著提升分类准确率。

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 只保留少量标签（约 10%）
np.random.seed(42)
n_labeled = int(len(y_train) * 0.10)
labeled_idx = np.random.choice(len(y_train), n_labeled, replace=False)
y_train_semi = np.full_like(y_train, fill_value=-1)
y_train_semi[labeled_idx] = y_train[labeled_idx]

print(f"=== 数据划分 ===")
print(f"训练集: {len(X_train)} 样本 (有标签: {(y_train_semi >= 0).sum()}, "
      f"无标签: {(y_train_semi == -1).sum()})")

### 2. 基准

运行 2. 基准 的代码，观察算法在该环节的行为。

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train[labeled_idx], y_train[labeled_idx])
print(f"仅用有标签数据 KNN: {knn.score(X_test, y_test):.4f}")

### 3. Label Spreading

运行 3. Label Spreading 的代码，观察算法在该环节的行为。

In [ ]:
# alpha: 正则化参数，控制保留多少原始标签信息 (范围 0 < alpha < 1)
# alpha→0: 完全传播（≈LabelPropagation）
# alpha→1: 完全保留原始标签（不传播）
ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=1000)
ls.fit(X_train, y_train_semi)

y_pred = ls.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\n=== 标签扩展 (alpha=0.2) ===")
print(f"测试准确率: {acc:.4f}")
print(f"迭代次数: {ls.n_iter_}")

### 4. alpha 参数影响

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== alpha 参数影响 ===")
for alpha in [0.001, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.999]:
    ls_a = LabelSpreading(kernel="knn", n_neighbors=7, alpha=alpha, max_iter=1000)
    ls_a.fit(X_train, y_train_semi)
    acc = ls_a.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  alpha={alpha}: 测试准确率={acc:.4f}, 迭代={ls_a.n_iter_}")

### 5. 不同标签比例

运行 5. 不同标签比例 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同标签比例对比 ===")
for ratio in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]:
    n_lab = int(len(y_train) * ratio)
    idx = np.random.choice(len(y_train), n_lab, replace=False)
    y_semi = np.full_like(y_train, fill_value=-1)
# --- 赋值/配置 ---
    y_semi[idx] = y_train[idx]

    ls_r = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2, max_iter=1000)
    ls_r.fit(X_train, y_semi)
    acc = ls_r.score(X_test, y_test)
    print(f"  标签比例={ratio:>4.0%}: 测试准确率={acc:.4f}")

### 6. 不同核参数

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== n_neighbors 对比 ===")
for nn in [3, 5, 7, 10, 15, 30]:
    ls_n = LabelSpreading(kernel="knn", n_neighbors=nn, alpha=0.2, max_iter=1000)
    ls_n.fit(X_train, y_train_semi)
    acc = ls_n.score(X_test, y_test)
# --- 输出结果 ---
    print(f"  n_neighbors={nn:>2}: 测试准确率={acc:.4f}")

### 7. 分类报告

在分类任务上训练模型并评估性能。

In [ ]:
print(f"\n=== 分类报告 ===")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

print("\n=== 标签扩展原理 ===")
print("Y = (1-α) × Y_propagated + α × Y_initial")
print("- α 控制 保持原始标签 vs 接受传播标签 的平衡")
print("- α 越小 → 传播越强，类边界越平滑")
print("- α 越大 → 越保守，接近只用原始标签")
# --- 输出结果 ---
print("- 比 LabelPropagation 更稳定（不容易出现数值问题）")

print("\n=== 标签扩展要点 ===")
print("- 与 LabelPropagation 的核心区别：alpha 正则化")
print("- alpha=0.2 是常用的默认值")
print("- 适合：数据有清晰的流形/聚类结构")
print("- 需要足够的有标签数据启动传播")

## 关键代码解释

### alpha 参数的刻度盘

```python
for alpha in [0.001, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 0.999]:
    ls_a = LabelSpreading(alpha=alpha)
```

alpha=0.001 接近纯传播（类似 LabelPropagation），alpha=0.999 几乎不传播（接近只用原始标签）。实验通常显示 alpha 在 0.1-0.3 之间效果最好。

### 标签比例实验

```python
for ratio in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]:
```

这个实验展示了半监督学习的核心价值：用 10% 的标签达到接近 100% 标签的效果。

## 使用示例

```python
from sklearn.semi_supervised import LabelSpreading
ls = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.2)
ls.fit(X, y_semi)
```

## 注意事项

1. **alpha=0.2 是常用默认值**
2. **标签太少时传播可能失效**：至少需要每个类别有一些标签
3. **与 LabelPropagation 的选择**：通常优先用 LabelSpreading（更稳定）

## 总结与延伸

以上代码展示了 **标签扩展** 的完整流程。

**解读要点**：
- 关注 **标签传播** 的准确率随迭代的变化
- 对比有监督 vs 半监督的性能差距
- 观察少量标签下的学习效果

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Deep Label Spreading**：用深度网络学习更好的相似度矩阵
- **Graph Laplacian 正则化**：LabelSpreading 的理论基础
- **协同训练（Co-Training）**：另一种半监督方法，用不同视角的特征分别训练
